# Week 16 Optional: Agentic AI for Data Engineering — Strands Agents & Amazon Bedrock AgentCore

## Overview

This optional notebook applies the **same Strands Agents + AgentCore** concepts
from the main Week 16 notebook to a **data engineering** domain. Instead of
investigating fraud, your agents will **monitor data pipelines, assess data
quality, and triage pipeline failures**.

The framework concepts are identical — what changes is the domain. This
reinforces the key lesson: agent patterns are portable across use cases.

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Build data engineering agents** with Strands Agents SDK
2. **Create a multi-agent pipeline monitoring system** using agents-as-tools
3. **Use AgentCore Memory** to persist pipeline investigation context
4. **See how the same agent patterns** apply across completely different domains

## Prerequisites

- Completed Week 16 main notebook (Strands agents, multi-agent, AgentCore Memory)
- Familiarity with data pipeline concepts (ETL, CDC, schema validation)

## Session Format (~2 hours)

| Section | Duration | Type |
|---------|----------|------|
| Section 0: Setup & Recap | 10 min | Code |
| Section 1: Strands Agents for Data Engineering | 20 min | Demo |
| Lab 1: Build a Pipeline Monitoring Agent | 15 min | Lab |
| Section 2: Multi-Agent Pipeline Monitoring | 25 min | Demo |
| Lab 2: Build a Multi-Agent Pipeline with Governance | 15 min | Lab |
| Section 3: AgentCore Memory for Pipeline Context | 15 min | Demo |
| Wrap-up & Homework | 5 min | Markdown |

## The Story

Your company runs dozens of data pipelines — nightly ETL jobs that load
data into the warehouse, hourly CDC syncs that replicate databases, weekly
aggregation jobs that build reporting tables. When a pipeline fails at 3 AM,
someone gets paged. But what if an AI agent could:

- **Triage the failure** — is it a transient error (retry) or a real problem?
- **Investigate the root cause** — check schema changes, upstream failures, volume anomalies
- **Decide the action** — auto-retry, flag for review, or escalate to on-call

That's what we'll build: a **multi-agent data pipeline monitoring system**
using Strands Agents + AgentCore.

## No GPU Needed

All work is API-based through Amazon Bedrock — no GPU required.

# Section 0: Environment Setup

We'll use **Strands Agents SDK** (AWS's open-source agent framework) and
the **Amazon Bedrock AgentCore SDK** for production memory services.

Both work seamlessly with Amazon Bedrock models — no additional API keys needed
beyond your AWS credentials from Week 13.

In [ ]:
# =============================================================================
# INSTALL REQUIRED LIBRARIES
# =============================================================================
# strands-agents: AWS's open-source agent framework
# strands-agents-tools: Pre-built tools (calculator, web search, etc.)
# bedrock-agentcore: AgentCore SDK (Memory, Runtime, Gateway)
# Note: boto3 and sagemaker are pre-installed on SageMaker

!pip install -q strands-agents strands-agents-tools bedrock-agentcore

# =============================================================================
# IMPORTS
# =============================================================================
import os
import json
import boto3
import sagemaker
from sagemaker import get_execution_role
from importlib.metadata import version
from strands import Agent, tool
from strands.models import BedrockModel
from strands.multiagent import Swarm              # Swarm pattern (Section 2 bonus)

# AgentCore Memory — low-level API
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole

# AgentCore Memory — Strands integration (auto stores/retrieves turns)
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

# =============================================================================
# VERIFY INSTALLATIONS
# =============================================================================
print("Library versions:")
print(f"  strands-agents:    {version('strands-agents')}")
print(f"  bedrock-agentcore: {version('bedrock-agentcore')}")
print(f"  boto3:             {boto3.__version__}")
print(f"  sagemaker:         {sagemaker.__version__}")
print("\nAll libraries installed successfully!")

In [ ]:
# =============================================================================
# SAGEMAKER + BEDROCK CONNECTION SETUP
# =============================================================================
# On SageMaker, we get AWS credentials automatically through the execution role.
# No need for getpass or manual credential entry — the IAM role attached to this
# notebook instance already has permissions for Bedrock.

sess = sagemaker.Session()
role = get_execution_role()
AWS_REGION = sess.boto_region_name

print(f"SageMaker execution role: {role.split('/')[-1]}")
print(f"AWS Region: {AWS_REGION}")

# Verify Bedrock connection
bedrock_client = boto3.client('bedrock', region_name=AWS_REGION)
try:
    models = bedrock_client.list_foundation_models()
    model_count = len(models['modelSummaries'])
    print(f"\n✅ Connected to Bedrock! {model_count} models available.")
except Exception as e:
    print(f"\n❌ Bedrock connection failed: {e}")
    print("Ask your instructor to verify the execution role has Bedrock permissions.")

# =============================================================================
# MODEL CONFIGURATION
# =============================================================================
# Same Claude Haiku model from Week 15 — fast, cheap, great at tool use
MODEL_ID = "us.anthropic.claude-3-haiku-20240307-v1:0"

# Create Strands BedrockModel using SageMaker's IAM role (automatic auth)
llm = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
)
print(f"\nAgent LLM: {MODEL_ID}")
print(f"Same Claude Haiku model from Week 15 — fast, cheap, supports tool use.")

# Section 1: Strands Agents for Data Engineering

## Same Framework, Different Domain

In the main notebook, you built a **fraud investigation agent**. Now we'll
build a **data pipeline monitoring agent** — same Strands SDK, same patterns,
completely different domain.

## How a Strands Agent Works (ReAct Loop)

```mermaid
graph LR
    User["Data Engineer"] -->|"RUN-001 failed"| Agent["Pipeline Agent"]
    Agent -->|"think"| LLM["Claude Haiku<br/>(Bedrock)"]
    LLM -->|"tool_use"| Router["Tool Router"]
    Router --> T1["lookup_pipeline_run"]
    Router --> T2["check_dataset_lineage"]
    Router --> T3["calculate_quality_score"]
    Router --> T4["check_data_governance_policy"]
    T1 -->|"result"| LLM
    T2 -->|"result"| LLM
    T3 -->|"result"| LLM
    T4 -->|"result"| LLM
    LLM -->|"final answer"| Agent
    Agent -->|"diagnosis + action"| User
```

This is the **same ReAct loop** from Week 15 and the main notebook:
Think → Act (call tool) → Observe (read result) → Repeat until done.

## Tool Mapping: Fraud vs Data Engineering

| Fraud Tool (Main Notebook) | Data Engineering Tool (This Notebook) |
|---|---|
| `lookup_transaction(txn_id)` | `lookup_pipeline_run(run_id)` |
| `check_customer_history(cust_id)` | `check_dataset_lineage(dataset_name)` |
| `calculate_risk_score(...)` | `calculate_quality_score(...)` |
| `check_fraud_policy(...)` | `check_data_governance_policy(...)` |

**Key insight**: The tools change, the pattern doesn't. This is what makes
agent skills transferable across domains.

In [ ]:
# =============================================================================
# DEMO: Build a Data Pipeline Monitoring Agent in Strands
# =============================================================================
# Same @tool pattern as the fraud agent — different domain, same framework.

# --- Tool 1: Look up a pipeline run ---
@tool
def lookup_pipeline_run(run_id: str) -> str:
    """Look up details of a data pipeline run by its run ID.

    Args:
        run_id: The pipeline run ID to look up (e.g., 'RUN-001')
    """
    runs = {
        "RUN-001": {
            "pipeline": "nightly_customers_etl",
            "status": "FAILED",
            "started_at": "2026-04-16 02:00:00",
            "failed_at": "2026-04-16 02:17:34",
            "rows_expected": 125000,
            "rows_processed": 0,
            "error": "SchemaValidationError: Column 'credit_score' expected INT, got VARCHAR",
            "source": "postgres://prod-db/customers",
            "target": "s3://data-lake/warehouse/customers/",
            "owner": "data-platform-team",
        },
        "RUN-002": {
            "pipeline": "hourly_transactions_cdc",
            "status": "SUCCESS",
            "started_at": "2026-04-16 14:00:00",
            "finished_at": "2026-04-16 14:03:22",
            "rows_expected": 12000,
            "rows_processed": 12450,
            "error": None,
            "source": "postgres://prod-db/transactions",
            "target": "s3://data-lake/warehouse/transactions/",
            "owner": "data-platform-team",
        },
        "RUN-042": {
            "pipeline": "weekly_reporting_aggregation",
            "status": "COMPLETED_WITH_WARNINGS",
            "started_at": "2026-04-14 06:00:00",
            "finished_at": "2026-04-14 06:45:12",
            "rows_expected": 500000,
            "rows_processed": 498712,
            "error": None,
            "warnings": [
                "3 columns have >5% null values: email, phone, middle_name",
                "Row count 0.26% below expected threshold",
            ],
            "source": "s3://data-lake/warehouse/*/",
            "target": "redshift://analytics/reporting_tables",
            "owner": "analytics-engineering",
        },
    }
    run = runs.get(run_id)
    if run:
        return json.dumps(run, indent=2)
    return f"Pipeline run {run_id} not found in monitoring database."

# --- Tool 2: Check dataset lineage ---
@tool
def check_dataset_lineage(dataset_name: str) -> str:
    """Retrieve upstream/downstream lineage and schema history for a dataset.

    Args:
        dataset_name: The dataset name to look up (e.g., 'customers')
    """
    lineage = {
        "customers": {
            "upstream": ["postgres://prod-db/customers"],
            "downstream": ["reporting_tables", "ml_feature_store", "customer_360"],
            "schema_version": "v3.2",
            "last_schema_change": "2026-04-15 (added credit_score as VARCHAR by app team)",
            "change_frequency": "~2 changes/month",
            "owner": "data-platform-team",
            "sla": "refreshed by 6:00 AM daily",
        },
        "transactions": {
            "upstream": ["postgres://prod-db/transactions"],
            "downstream": ["reporting_tables", "fraud_detection_model"],
            "schema_version": "v5.1",
            "last_schema_change": "2026-03-01 (no recent changes)",
            "change_frequency": "~1 change/quarter",
            "owner": "data-platform-team",
            "sla": "refreshed hourly, <5 min latency",
        },
    }
    info = lineage.get(dataset_name)
    if info:
        return json.dumps(info, indent=2)
    return f"Dataset '{dataset_name}' not found in lineage catalog."

# --- Tool 3: Calculate data quality score ---
@tool
def calculate_quality_score(null_percentage: float, schema_drift: bool,
                            freshness_hours: float, volume_deviation_pct: float) -> str:
    """Calculate a data quality score based on key quality dimensions.

    Args:
        null_percentage: Percentage of null values across key columns (0-100)
        schema_drift: Whether the schema has changed unexpectedly
        freshness_hours: How many hours since the last successful refresh
        volume_deviation_pct: Percentage deviation from expected row count
    """
    score = 100
    factors = []
    if null_percentage > 10:
        score -= 30
        factors.append(f"High null rate ({null_percentage:.1f}%)")
    elif null_percentage > 5:
        score -= 15
        factors.append(f"Moderate null rate ({null_percentage:.1f}%)")
    if schema_drift:
        score -= 30
        factors.append("Unexpected schema change detected")
    if freshness_hours > 24:
        score -= 25
        factors.append(f"Stale data ({freshness_hours:.0f}h since refresh)")
    elif freshness_hours > 12:
        score -= 10
        factors.append(f"Aging data ({freshness_hours:.0f}h since refresh)")
    if volume_deviation_pct > 10:
        score -= 20
        factors.append(f"Volume anomaly ({volume_deviation_pct:.1f}% deviation)")
    elif volume_deviation_pct > 5:
        score -= 10
        factors.append(f"Minor volume deviation ({volume_deviation_pct:.1f}%)")
    score = max(0, score)
    level = "HEALTHY" if score >= 80 else "DEGRADED" if score >= 50 else "CRITICAL"
    return json.dumps({
        "quality_score": score,
        "quality_level": level,
        "factors": factors if factors else ["All quality checks passed"],
    }, indent=2)

# --- Tool 4: Check data governance policy ---
@tool
def check_data_governance_policy(quality_level: str, dataset_name: str,
                                  has_pii: bool) -> str:
    """Look up the data governance policy for a given quality level and dataset.

    Args:
        quality_level: Quality level — HEALTHY, DEGRADED, or CRITICAL
        dataset_name: The dataset being evaluated
        has_pii: Whether the dataset contains PII (personally identifiable information)
    """
    policies = {
        "HEALTHY": "No action needed. Log quality metrics for trending.",
        "DEGRADED": "Flag for data engineering review within 24 hours. "
                    "Notify downstream consumers via Slack.",
        "CRITICAL": "Halt downstream pipelines immediately. Page on-call data engineer. "
                    "Create incident ticket.",
    }
    policy = policies.get(quality_level.upper(), "Unknown quality level")
    pii_note = ""
    if has_pii:
        pii_note = ("PII ALERT: Dataset contains PII. Any quality issues "
                    "must be reported to the Data Privacy Officer within 4 hours.")
    return json.dumps({
        "policy": policy,
        "pii_handling": pii_note if pii_note else "No PII detected — standard handling.",
        "escalation": quality_level == "CRITICAL",
        "downstream_impact": f"Check downstream consumers of '{dataset_name}'",
    }, indent=2)

# --- Create the agent ---
pipeline_tools = [lookup_pipeline_run, check_dataset_lineage,
                  calculate_quality_score, check_data_governance_policy]

pipeline_agent = Agent(
    model=llm,
    tools=pipeline_tools,
    system_prompt=(
        "You are a data pipeline monitoring agent. "
        "When given a pipeline run to investigate, use your tools to: "
        "1) Look up the pipeline run details, "
        "2) Check the dataset lineage and schema history, "
        "3) Calculate the data quality score, "
        "4) Check the data governance policy. "
        "Then provide a clear diagnosis with your recommended action."
    ),
)

print("Pipeline monitoring agent created with Strands!")
print(f"   Model: {MODEL_ID}")
print(f"   Tools: {[t.tool_name for t in pipeline_tools]}")
print(f"\n   Same pattern as the fraud agent — different tools, same framework.")

In [ ]:
# =============================================================================
# DEMO: Run the Pipeline Agent on a Failed ETL Job
# =============================================================================

print("Investigating RUN-001 (failed nightly ETL)...")
print("=" * 60)

response = pipeline_agent(
    "Investigate pipeline run RUN-001. The nightly customers ETL failed "
    "with 0 rows loaded. Determine the root cause, assess data quality "
    "impact, and recommend an action."
)

print("=" * 60)
print(f"\nAgent's response:\n{response}")

In [ ]:
# =============================================================================
# DEMO: Run on a Healthy Pipeline — Compare Agent Behavior
# =============================================================================

print("Investigating RUN-002 (hourly CDC sync — healthy)...")
print("=" * 60)

healthy_response = pipeline_agent(
    "Check pipeline run RUN-002. The hourly transactions CDC sync "
    "completed — is everything looking good?"
)

print("=" * 60)
print(f"\nAgent's response:\n{healthy_response}")
print(f"\n   Notice: for healthy pipelines, the agent should confirm all-clear")
print(f"   quickly — no need for deep investigation.")

> **Think About It**: You just built a data pipeline monitoring agent with the
> same framework you used for fraud investigation. The tools changed but the
> pattern didn't. In your organization, what other domains could benefit from
> this agent pattern? Think about IT operations, security monitoring, customer
> support — anywhere you have "investigate and decide" workflows.

## Lab 1: Build a Pipeline Monitoring Agent with Strands (15 minutes)

### Your Task

Build a Strands agent with a NEW tool and a customized system prompt for
diagnosing pipeline failures.

### Steps

1. **Create a new tool** called `get_similar_failures` that takes an
   `error_type` (string) and returns a list of recent pipeline runs that
   had similar errors. Use the `@tool` decorator with full docstring
   (including Args section).
2. **Create an Agent** with ALL 5 tools (the 4 from the demo + your new one),
   a Bedrock model, and a system prompt that instructs the agent to always
   check for similar past failures before recommending an action.
3. **Test your agent** on a pipeline with a schema error — does it use
   your new tool to find patterns?

### Homework Extension

After class: add a 6th tool called `check_data_freshness` that queries
a simulated metadata catalog and returns the last refresh time, expected
refresh schedule, and whether the dataset is currently stale. Integrate it
into your agent.

In [ ]:
# =============================================================================
# SOLUTION: LAB 1 — BUILD A PIPELINE MONITORING AGENT WITH STRANDS
# =============================================================================

# Step 1: Create the get_similar_failures tool
@tool
def get_similar_failures(error_type: str) -> str:
    """Find recent pipeline runs that had similar error patterns.

    Args:
        error_type: The type of error to search for (e.g., 'SchemaValidationError')
    """
    # Simulated failure history database
    failure_history = {
        "SchemaValidationError": [
            {"run_id": "RUN-887", "pipeline": "nightly_customers_etl",
             "date": "2026-04-02", "resolution": "App team changed column type without notice — updated ETL mapping"},
            {"run_id": "RUN-654", "pipeline": "daily_products_sync",
             "date": "2026-03-15", "resolution": "New column added upstream — added to schema registry"},
        ],
        "ConnectionTimeout": [
            {"run_id": "RUN-901", "pipeline": "hourly_transactions_cdc",
             "date": "2026-04-10", "resolution": "Database failover in progress — auto-retried successfully"},
            {"run_id": "RUN-899", "pipeline": "hourly_transactions_cdc",
             "date": "2026-04-10", "resolution": "Same failover event — resolved after 15 min"},
        ],
        "VolumeAnomaly": [
            {"run_id": "RUN-812", "pipeline": "weekly_reporting_aggregation",
             "date": "2026-03-28", "resolution": "Marketing campaign caused 3x normal volume — adjusted thresholds"},
        ],
    }

    matches = failure_history.get(error_type, [])
    return json.dumps({
        "error_type": error_type,
        "similar_failures": matches,
        "total_count": len(matches),
        "pattern_note": f"Found {len(matches)} similar failures — check resolutions for patterns"
                        if matches else "No similar failures found — this may be a new issue type",
    }, indent=2)


# Step 2: Create an Agent with all 5 tools
all_tools = [lookup_pipeline_run, check_dataset_lineage,
             calculate_quality_score, check_data_governance_policy,
             get_similar_failures]

lab1_agent = Agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You are a senior data pipeline investigator. "
        "When investigating a pipeline issue, ALWAYS: "
        "1) Look up the pipeline run details, "
        "2) Check the dataset lineage and schema history, "
        "3) Search for similar past failures to identify patterns, "
        "4) Calculate the data quality score, "
        "5) Check the data governance policy. "
        "Provide a detailed diagnosis with evidence from ALL tools."
    ),
)

# Step 3: Test the agent
print("Investigating pipeline with warnings...")
print("=" * 60)
test_response = lab1_agent(
    "Pipeline weekly_reporting_aggregation completed with warnings — "
    "3 columns have high null rates. Check if this is a recurring pattern "
    "and assess the impact."
)
print("=" * 60)
print(f"\n{test_response}")
print("\nLab 1 complete!")

# Section 2: Multi-Agent Pipeline Monitoring

## The Agents-as-Tools Pattern

In production data platforms, you don't have ONE agent doing everything.
You have **specialized agents** that each handle a piece of the pipeline
monitoring workflow:

```mermaid
graph TD
    DE["Data Engineer"] -->|"RUN-001 failed"| Sup["Supervisor Agent"]
    Sup -->|"1. Classify severity"| Triage["Triage Agent<br/>lookup_pipeline_run<br/>calculate_quality_score"]
    Triage -->|"severity: CRITICAL"| Sup
    Sup -->|"2. Gather evidence"| Investigate["Investigation Agent<br/>check_dataset_lineage<br/>get_similar_failures"]
    Investigate -->|"root cause report"| Sup
    Sup -->|"3. Decide action"| Remediate["Remediation Agent<br/>check_data_governance_policy"]
    Remediate -->|"action: ESCALATE_TO_ONCALL"| Sup
    Sup -->|"final diagnosis"| DE

    style Triage fill:#f9d71c,stroke:#333
    style Investigate fill:#4ecdc4,stroke:#333
    style Remediate fill:#ff6b6b,stroke:#333
```

**How it works in Strands**:
1. Each specialist is a regular `Agent()` with its own tools and system prompt
2. Wrap each specialist in a `@tool` decorator — now it's callable like a function
3. Give all specialist-tools to a supervisor `Agent()`
4. The supervisor decides which specialists to call and in what order

This is called the **agents-as-tools** pattern — same concept as the main
notebook's fraud pipeline, applied to data engineering.

In [ ]:
# =============================================================================
# DEMO: Define Specialist Agents for the Pipeline Monitoring System
# =============================================================================

# --- Specialist 1: Triage Agent ---
triage_agent = Agent(
    model=llm,
    tools=[lookup_pipeline_run, calculate_quality_score],
    system_prompt=(
        "You are a data pipeline triage specialist. Your ONLY job is to quickly "
        "classify pipeline issues by severity. "
        "Look up the pipeline run, calculate the quality score, and return "
        "a JSON-formatted triage result with: run_id, severity "
        "(LOW/MEDIUM/HIGH), quality_score, and key_factors. "
        "Be concise — no lengthy analysis. Speed matters in triage."
    ),
    callback_handler=None,
)

# --- Specialist 2: Investigation Agent ---
investigation_agent = Agent(
    model=llm,
    tools=[lookup_pipeline_run, check_dataset_lineage,
           get_similar_failures],
    system_prompt=(
        "You are a data pipeline investigation specialist. You receive pipeline "
        "runs that have been flagged as MEDIUM or HIGH severity. Your job is to: "
        "1) Gather all available evidence (run details, dataset lineage, "
        "   similar past failures) "
        "2) Write a detailed root cause analysis report "
        "3) Note any upstream schema changes or recurring patterns "
        "Be thorough — your report will be used for the remediation decision."
    ),
    callback_handler=None,
)

# --- Specialist 3: Remediation Agent ---
remediation_agent = Agent(
    model=llm,
    tools=[check_data_governance_policy],
    system_prompt=(
        "You are a data pipeline remediation specialist. You receive a triage "
        "result and an investigation report. Your job is to: "
        "1) Check the data governance policy for the given severity and dataset "
        "2) Render a FINAL ACTION: AUTO_RETRY, FLAG_FOR_REVIEW, or ESCALATE_TO_ONCALL "
        "3) Provide a one-paragraph justification "
        "4) List the recommended remediation steps "
        "Your decision must be defensible in a post-incident review."
    ),
    callback_handler=None,
)

print("Three specialist agents created:")
print(f"   1. Triage Agent      — tools: lookup_pipeline_run, calculate_quality_score")
print(f"   2. Investigation     — tools: lookup_pipeline_run, check_dataset_lineage, get_similar_failures")
print(f"   3. Remediation Agent — tools: check_data_governance_policy")
print(f"\n   callback_handler=None suppresses each agent's intermediate output.")
print(f"   Only the supervisor's output will be visible to the user.")

In [ ]:
# =============================================================================
# DEMO: Wrap Specialist Agents as Tools & Create Supervisor
# =============================================================================

@tool
def run_triage(run_id: str) -> str:
    """Run pipeline triage to quickly classify a run's issue severity.

    Args:
        run_id: The pipeline run ID to triage
    """
    result = triage_agent(
        f"Triage pipeline run {run_id}. Classify the severity."
    )
    return str(result)

@tool
def run_investigation(run_id: str, dataset_name: str,
                      triage_result: str) -> str:
    """Run a deep investigation on a flagged pipeline run.

    Args:
        run_id: The pipeline run ID to investigate
        dataset_name: The primary dataset affected
        triage_result: The triage result with severity and factors
    """
    result = investigation_agent(
        f"Investigate pipeline run {run_id} affecting dataset '{dataset_name}'. "
        f"Triage result: {triage_result}"
    )
    return str(result)

@tool
def run_remediation(triage_result: str, investigation_report: str,
                    dataset_name: str) -> str:
    """Decide remediation action based on triage and investigation findings.

    Args:
        triage_result: The triage classification result
        investigation_report: The detailed investigation report
        dataset_name: The primary dataset affected
    """
    result = remediation_agent(
        f"Decide remediation. Triage: {triage_result}. "
        f"Investigation: {investigation_report}. Dataset: {dataset_name}."
    )
    return str(result)

# --- Create the Supervisor Agent ---
supervisor = Agent(
    model=llm,
    tools=[run_triage, run_investigation, run_remediation],
    system_prompt=(
        "You are the data platform operations supervisor. "
        "You coordinate pipeline incident response by delegating to specialist agents: "
        "1) ALWAYS start with run_triage to classify the severity "
        "2) If severity is MEDIUM or HIGH, run_investigation to gather evidence "
        "3) If severity is LOW, skip investigation and go straight to run_remediation "
        "4) ALWAYS end with run_remediation for the final action "
        "Report the final diagnosis and action clearly to the user."
    ),
)

print("Supervisor agent created with 3 specialist agents as tools!")
print(f"   Tools: run_triage, run_investigation, run_remediation")
print(f"\n   Flow: Supervisor -> Triage -> (if needed) Investigation -> Remediation")

In [ ]:
# =============================================================================
# DEMO: Run the Full Multi-Agent Pipeline Monitoring System
# =============================================================================

print("Running multi-agent investigation on RUN-001 (failed ETL)...")
print("=" * 60)

supervisor_response = supervisor(
    "Pipeline run RUN-001 failed — the nightly customers ETL loaded 0 rows "
    "due to a SchemaValidationError. The customers dataset has 3 downstream "
    "consumers. Run the full investigation pipeline."
)

print("=" * 60)
print(f"\nSupervisor's final report:\n{supervisor_response}")

In [ ]:
# =============================================================================
# DEMO: Run on a Healthy Pipeline
# =============================================================================

print("Running multi-agent check on RUN-002 (hourly CDC sync — healthy)...")
print("=" * 60)

healthy_response = supervisor(
    "Check pipeline run RUN-002 — the hourly transactions CDC sync "
    "completed successfully with 12,450 rows."
)

print("=" * 60)
print(f"\nSupervisor's final report:\n{healthy_response}")
print(f"\n   Notice: for HEALTHY pipelines, the supervisor should skip investigation")
print(f"   and go directly to remediation (which confirms no action needed).")

## Bonus: The Swarm Pattern — Decentralized Collaboration

The **agents-as-tools** pattern above uses a supervisor to control the flow.
The **Swarm** pattern is the opposite: agents hand off to each other freely.

| | Supervisor (agents-as-tools) | Swarm |
|---|---|---|
| **Control** | Central supervisor decides flow | Each agent decides who's next |
| **Flow** | Fixed: Triage -> Investigate -> Remediate | Emergent: agents hand off freely |
| **Best for** | Predictable pipelines, audit trails | Exploration, complex root cause analysis |
| **Trade-off** | More control, less flexible | More flexible, harder to predict |

In [ ]:
# =============================================================================
# BONUS DEMO: Swarm Pattern — Pipeline Agents, Decentralized Flow
# =============================================================================

swarm_triage = Agent(
    name="triage",
    model=llm,
    tools=[lookup_pipeline_run, calculate_quality_score],
    system_prompt=(
        "You are a pipeline triage specialist. Look up the pipeline run and "
        "calculate the quality score. "
        "If severity is MEDIUM or HIGH, hand off to 'investigator' with your findings. "
        "If severity is LOW, hand off directly to 'remediator' with your findings."
    ),
)

swarm_investigator = Agent(
    name="investigator",
    model=llm,
    tools=[lookup_pipeline_run, check_dataset_lineage, get_similar_failures],
    system_prompt=(
        "You are a pipeline investigator. Check dataset lineage and search "
        "for similar past failures. Write a root cause analysis report. "
        "When done, hand off to 'remediator' with your report."
    ),
)

swarm_remediator = Agent(
    name="remediator",
    model=llm,
    tools=[check_data_governance_policy],
    system_prompt=(
        "You are the remediation specialist. Review triage and investigation "
        "findings, check the governance policy, and decide: AUTO_RETRY, "
        "FLAG_FOR_REVIEW, or ESCALATE_TO_ONCALL. "
        "When decided, call complete_swarm_task with your action."
    ),
)

pipeline_swarm = Swarm(
    [swarm_triage, swarm_investigator, swarm_remediator],
    entry_point=swarm_triage,
    max_handoffs=10,
)

print("Swarm investigating RUN-001 (decentralized flow)...")
print("=" * 60)

swarm_result = pipeline_swarm(
    "Pipeline run RUN-001 failed — nightly customers ETL loaded 0 rows "
    "due to a SchemaValidationError. Investigate and decide action."
)

print("=" * 60)
print(f"\nSwarm status: {swarm_result.status}")
print(f"Agent flow: {' -> '.join(node.node_id for node in swarm_result.node_history)}")
print(f"\nThe agents decided the investigation flow themselves!")

> **Think About It**: The supervisor agent decided whether to skip investigation
> based on the triage severity. For data pipelines, some teams use hard-coded
> rules for routing (e.g., "if status == FAILED, always investigate"). Others
> let the LLM decide based on context. What approach would you prefer for
> production pipeline monitoring? When might the LLM's judgment add value
> over simple rules?

## Lab 2: Build a Multi-Agent Pipeline with Governance (15 minutes)

### Your Task

Extend the multi-agent pipeline with a 4th specialist agent: a **Governance
Agent** that checks data regulatory compliance.

### Steps

1. **Create a governance agent** — a specialist that checks whether a
   pipeline issue involves regulatory concerns (e.g., GDPR right-to-deletion
   requests pending, CCPA data retention violations, PII exposure in logs).
   Give it a `@tool` called `check_regulatory_rules`.
2. **Wrap it as a tool** called `run_governance_check` using the agents-as-tools pattern.
3. **Update the supervisor** — add the governance tool and update the system
   prompt so the supervisor always runs governance check after investigation
   but before remediation.
4. **Test the full pipeline** on RUN-001 (customers ETL failed — customers
   dataset contains PII). Does the governance agent flag the PII concern?

### Homework Extension

After class: add error handling to the pipeline. What happens if the
triage agent fails or returns an unexpected severity? Implement a fallback
that routes directly to on-call when any agent in the pipeline errors.

In [ ]:
# =============================================================================
# SOLUTION: LAB 2 — MULTI-AGENT PIPELINE MONITORING WITH GOVERNANCE
# =============================================================================

# Step 1: Create regulatory rules tool
@tool
def check_regulatory_rules(dataset_name: str, has_pii: bool,
                           data_retention_days: int) -> str:
    """Check regulatory compliance rules for a dataset involved in a pipeline issue.

    Args:
        dataset_name: The dataset being evaluated
        has_pii: Whether the dataset contains PII
        data_retention_days: How many days the data has been retained
    """
    violations = []
    requirements = []

    # GDPR: Right to deletion must be honored within 30 days
    if has_pii:
        requirements.append("GDPR: Verify no pending right-to-deletion requests for this dataset")
        requirements.append("GDPR: Data breach notification required within 72 hours if PII exposed")
        if data_retention_days > 365:
            violations.append(f"GDPR: Data retained {data_retention_days} days — exceeds 365-day policy")

    # CCPA: California consumer data retention limits
    if has_pii and data_retention_days > 730:
        violations.append(f"CCPA: Data retained {data_retention_days} days — exceeds 2-year limit")

    # Data quality governance: stale PII data is a risk
    if has_pii:
        requirements.append("PII datasets with quality issues must be reported to Data Privacy Officer within 4 hours")

    # General data governance
    requirements.append(f"Document incident in data governance log for '{dataset_name}'")

    return json.dumps({
        "compliant": len(violations) == 0,
        "violations": violations,
        "requirements": requirements,
        "action": "BLOCK — governance violation" if violations else "PROCEED with requirements",
    }, indent=2)


# Step 2: Create the governance agent
governance_agent = Agent(
    model=llm,
    tools=[check_regulatory_rules],
    system_prompt=(
        "You are a data governance specialist. "
        "Check all applicable regulations for the dataset in question: "
        "GDPR, CCPA, data retention policies, PII handling requirements. "
        "Return a clear governance assessment with any violations or requirements."
    ),
    callback_handler=None,
)

# Step 3: Wrap as tool
@tool
def run_governance_check(dataset_name: str, has_pii: bool,
                         data_retention_days: int) -> str:
    """Run regulatory governance check on a dataset involved in a pipeline issue.

    Args:
        dataset_name: The dataset being evaluated
        has_pii: Whether the dataset contains PII
        data_retention_days: How many days the data has been retained
    """
    result = governance_agent(
        f"Check governance for dataset '{dataset_name}'. "
        f"Contains PII: {has_pii}. Data retained: {data_retention_days} days."
    )
    return str(result)

# Step 4: Create updated supervisor with governance
updated_supervisor = Agent(
    model=llm,
    tools=[run_triage, run_investigation, run_governance_check, run_remediation],
    system_prompt=(
        "You are the data platform operations supervisor. "
        "Coordinate the full pipeline incident response: "
        "1) ALWAYS start with run_triage to classify severity "
        "2) If severity is MEDIUM or HIGH, run_investigation for evidence "
        "3) ALWAYS run_governance_check regardless of severity "
        "4) ALWAYS end with run_remediation for the final action, "
        "   including governance findings in your decision context "
        "Report the complete findings to the user."
    ),
)

# Step 5: Test
print("Full pipeline with governance check on RUN-001...")
print("=" * 60)
test_result = updated_supervisor(
    "Run the full pipeline investigation on RUN-001. "
    "The nightly customers ETL failed — the customers dataset contains PII "
    "and has been retained for 400 days. Include governance check."
)
print("=" * 60)
print(f"\n{test_result}")
print("\nLab 2 complete!")

# Section 3: AgentCore Memory — Persistent Pipeline Knowledge

## From Ephemeral to Persistent Memory

In Week 15, we used LangGraph's `MemorySaver`:
- Simple: `MemorySaver()` + `thread_id`
- **Ephemeral**: Memory dies when the notebook restarts
- **In-process**: Can't share memory between agents or services
- **No long-term learning**: Each session starts from scratch

**Amazon Bedrock AgentCore Memory** solves all of these:
- **Persistent**: Stored in AWS — survives restarts, deployments, scaling
- **Shared**: Any agent (or service) can read/write to the same memory
- **Short-term + Long-term**: Session turns (conversation) + extracted knowledge
- **Searchable**: Semantic search over stored memories
- **Managed**: No database to maintain, automatic scaling

## Memory Types for Data Engineering

| Type | What it stores | Data Engineering Example |
|------|---------------|------------------------|
| **Short-term** | Turn-by-turn conversation | "What pipeline run are we investigating?" |
| **Long-term (Semantic)** | Extracted entities and facts | "The customers table schema changes frequently" |
| **Long-term (Summary)** | Session summaries | "Last incident: schema drift in customers ETL, resolved by app team" |
| **Long-term (Episodic)** | Structured episodes | "RUN-001 was escalated due to PII impact + schema drift" |

In [ ]:
# =============================================================================
# DEMO: AgentCore Memory — Pipeline Agent That Remembers Across Turns
# =============================================================================
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=AWS_REGION)
control_client = boto3.client('bedrock-agentcore-control', region_name=AWS_REGION)
MEMORY_ID = None
MEMORY_NAME = "week16-pipeline-investigation"

try:
    memories = control_client.list_memories().get('memories', [])
    existing = next((m for m in memories if m.get('status') == 'ACTIVE'), None)
    if existing:
        MEMORY_ID = existing['id']
        print(f"Found existing ACTIVE memory (id: {MEMORY_ID})")
    else:
        print(f"Creating memory '{MEMORY_NAME}' (this takes 1-2 minutes)...")
        memory = memory_client.create_memory_and_wait(
            name=MEMORY_NAME,
            description="Pipeline investigation memory for Week 16",
            event_expiry_days=1,
        )
        MEMORY_ID = memory['id']
        print(f"Created memory (id: {MEMORY_ID}, status: ACTIVE)")
except Exception as e:
    print(f"AgentCore Memory not available: {e}")

if MEMORY_ID:
    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id="pipeline-investigation-demo",
        actor_id="data-engineer",
    )
    with AgentCoreMemorySessionManager(
        agentcore_memory_config=config, region_name=AWS_REGION,
    ) as memory_session:
        memory_agent = Agent(
            model=llm, tools=pipeline_tools,
            system_prompt=(
                "You are a data pipeline monitoring agent with persistent memory. "
                "When investigating, use your tools thoroughly. "
                "Remember findings from previous turns in this session."
            ),
            session_manager=memory_session,
        )
        print("TURN 1: Investigating RUN-001...")
        print("=" * 60)
        result1 = memory_agent("Investigate pipeline run RUN-001. What failed and why?")
        print(str(result1)[:500])

        print("\n\nTURN 2: Follow-up (agent should remember)...")
        print("=" * 60)
        result2 = memory_agent(
            "What was the quality level of the pipeline we just investigated? "
            "What governance policy applies?"
        )
        print(str(result2)[:500])
    print(f"\n✅ Both turns stored in AgentCore Memory!")
    print(f"   Memory ID: {MEMORY_ID}")
    print(f"   This persists across notebook restarts.")
else:
    print("\nCode pattern (for reference):")
    print("  client = MemoryClient(region_name=AWS_REGION)")
    print("  memory = client.create_memory_and_wait(name='my-memory', event_expiry_days=1)")
    print("  config = AgentCoreMemoryConfig(memory_id=memory['id'], session_id='s1', actor_id='a1')")
    print("  with AgentCoreMemorySessionManager(config, region_name=AWS_REGION) as sm:")
    print("      agent = Agent(model=llm, tools=pipeline_tools, session_manager=sm)")
    print("      agent('Investigate RUN-001')  # auto-stored")

In [ ]:
# =============================================================================
# DEMO: The Full AgentCore Production Stack for Data Engineering
# =============================================================================

print("Amazon Bedrock AgentCore — Data Engineering Production Architecture")
print("=" * 60)
print()
print("See the Mermaid diagram in the markdown cell below for the visual.")
print()
print("AgentCore Services for Data Engineering:")
print()
print("  1. Runtime    — Host your pipeline monitoring agents in managed containers")
print("                  Auto-scales with incident volume")
print()
print("  2. Memory     — Persistent knowledge across investigations")
print("                  'Last time this pipeline failed, root cause was X'")
print()
print("  3. Gateway    — Connect agents to your existing APIs:")
print("                  Airflow REST API, dbt Cloud API, Snowflake query API")
print("                  Auto-generates MCP tools from OpenAPI specs")
print()
print("  4. Identity   — Secure credential management")
print("                  Agents get scoped access to databases and APIs")
print()
print("  5. Observability — Trace agent decisions for post-incident reviews")
print("                     'Why did the agent escalate RUN-001 but retry RUN-042?'")

> **Think About It**: We built a multi-agent pipeline monitoring system in a
> notebook. In production, this system would integrate with Airflow, dbt,
> Snowflake, PagerDuty, and Slack. Which AgentCore services (Runtime, Memory,
> Gateway, Identity, Observability) would be most critical to set up first
> for a data engineering use case? Why might the priorities differ from a
> fraud detection use case?

```mermaid
graph TB
    subgraph "AgentCore Runtime"
        Sup["Supervisor Agent"] --> T["Triage Agent"]
        Sup --> I["Investigation Agent"]
        Sup --> R["Remediation Agent"]
    end
    Sup --- Mem["AgentCore Memory<br/>(past incidents)"]
    Sup --- GW["AgentCore Gateway<br/>(Airflow, dbt, Snowflake APIs)"]
    Sup --- ID["AgentCore Identity<br/>(DB credentials)"]
    Sup --- OBS["AgentCore Observability<br/>(decision traces)"]

    style Mem fill:#4ecdc4,stroke:#333
    style GW fill:#f9d71c,stroke:#333
    style ID fill:#ff6b6b,stroke:#333
    style OBS fill:#a78bfa,stroke:#333
```

## Connecting the Dots: Pipeline Agents + ML Models

Remember Week 14's fine-tuned models? In a production data engineering
system, you could use a fine-tuned model as a **fast anomaly classifier**:

```python
@tool
def classify_anomaly_with_model(pipeline_metrics: str) -> str:
    """Run the fine-tuned anomaly detection model on pipeline metrics."""
    from transformers import pipeline
    classifier = pipeline("text-classification", model="./pipeline-anomaly-detector")
    result = classifier(pipeline_metrics)[0]
    return json.dumps({"prediction": result["label"], "confidence": result["score"]})
```

**The production architecture**:
1. Fine-tuned model classifies pipeline health first (fast, runs locally)
2. Only DEGRADED/CRITICAL cases go to the full multi-agent investigation
3. This saves API costs — the LLM agents only handle complex incidents

This is a homework extension — try it after class!

## Looking Ahead

- **Weeks 17-18**: RAG (Retrieval-Augmented Generation) — give agents access to
  documentation knowledge bases. Imagine your pipeline agent searching through
  runbooks and past incident reports.
- **Weeks 19-20**: MLOps — how to version, monitor, and maintain your agent
  pipelines in production.

# Summary: What We Learned Today

## Key Takeaways

### Same Patterns, Different Domain
- Built a complete data pipeline monitoring system using Strands Agents SDK
- **Tools changed** (fraud → data engineering), **patterns stayed the same**
- This proves agent skills are transferable across domains

### Data Engineering Agent Tools
- `lookup_pipeline_run` — get run status, errors, row counts
- `check_dataset_lineage` — trace upstream/downstream dependencies
- `calculate_quality_score` — score data quality (nulls, schema drift, freshness, volume)
- `check_data_governance_policy` — lookup SLA and PII handling rules

### Multi-Agent Pipeline Monitoring
- **Triage Agent** — fast severity classification (HEALTHY/DEGRADED/CRITICAL)
- **Investigation Agent** — root cause analysis (lineage, schema changes, past failures)
- **Remediation Agent** — decide action (AUTO_RETRY, FLAG_FOR_REVIEW, ESCALATE_TO_ONCALL)
- **Supervisor** — orchestrates the pipeline, skips investigation for healthy runs

### AgentCore Memory for Incident Context
- Persistent memory lets agents learn from past incidents
- "Last time this pipeline failed, it was a schema change by the app team"
- Shared across agents — the remediation agent can see what triage found

# Homework

## Homework 1: Anomaly Classifier as Agent Tool
Wrap a simple anomaly detection model as a Strands `@tool`. Build a pipeline
where the model pre-screens pipeline metrics and only sends anomalous cases
to the full multi-agent investigation system.

## Homework 2: Memory-Enriched Pipeline Agent
If you have access to AgentCore Memory, build an agent that:
- Stores pipeline investigation results in AgentCore Memory
- Before investigating a new failure, searches memory for similar past incidents
- Uses past root causes to speed up current investigation

# Great Work!

You've completed the optional data engineering notebook for Week 16.

**What you proved today**: The agent patterns from fraud detection — ReAct loops,
tools, multi-agent orchestration, persistent memory — apply directly to data
engineering. Learn the pattern once, apply it everywhere.